<a href="https://colab.research.google.com/github/omark243/Big-Data-/blob/main/bigdata_assignment_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Assignment Solution

This notebook loads the CSV file into a Spark DataFrame and answers the required questions using PySpark DataFrame API.


In [ ]:
# Install required libraries in Google Colab
!pip install -q pyspark gdown


In [ ]:
# Download CSV file from Google Drive
# File ID from the assignment link
file_id = "1GGWUxXhYCUYnmnYZK4h962liz2omQews"
!gdown "https://drive.google.com/uc?id={file_id}" -O data.csv


Downloading...
From: https://drive.google.com/uc?id=1GGWUxXhYCUYnmnYZK4h962liz2omQews
To: /content/data.csv
100% 5.89k/5.89k [00:00<00:00, 13.0MB/s]


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round

# Create Spark session
spark = SparkSession.builder.appName("BigData_Assignment").getOrCreate()

# Load CSV into Spark DataFrame
df = spark.read.csv("data.csv", header=True, inferSchema=True)

# Check schema and first rows
df.printSchema()
df.show(5)


root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)

+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|                        5|        

## 1. Count the number of employees in each city


In [ ]:
q1 = df.groupBy("City") \
       .agg(count("*").alias("NumberOfEmployees")) \
       .orderBy("City")

q1.show()


+---------+-----------------+
|     City|NumberOfEmployees|
+---------+-----------------+
|Bangalore|               73|
|New Delhi|               40|
|     Pune|               37|
+---------+-----------------+



## 2. Calculate the average ExperienceInCurrentDomain for each Education level


In [ ]:
q2 = df.groupBy("Education") \
       .agg(round(avg("ExperienceInCurrentDomain"), 2).alias("AvgExperienceInCurrentDomain")) \
       .orderBy("Education")

q2.show()


+---------+----------------------------+
|Education|AvgExperienceInCurrentDomain|
+---------+----------------------------+
|Bachelors|                        2.45|
|  Masters|                        2.91|
|      PHD|                        3.11|
+---------+----------------------------+



## 3. Calculate the average age of employees in each city


In [ ]:
q3 = df.groupBy("City") \
       .agg(round(avg("Age"), 2).alias("AverageAge")) \
       .orderBy("City")

q3.show()


+---------+----------+
|     City|AverageAge|
+---------+----------+
|Bangalore|     29.38|
|New Delhi|     29.15|
|     Pune|     29.11|
+---------+----------+



## 4. How many female employees have worked 2 years or more in the company?


In [ ]:
# Using ExperienceInCurrentDomain as the available experience column
q4 = df.filter((col("Gender") == "Female") & (col("ExperienceInCurrentDomain") >= 2)).count()

print("Number of female employees with 2 years or more experience:", q4)


Number of female employees with 2 years or more experience: 38


## 5. How many employees who are more than 27 years old have left the company?


In [ ]:
q5 = df.filter((col("Age") > 27) & (col("LeaveOrNot") == 1)).count()

print("Number of employees older than 27 who left the company:", q5)


Number of employees older than 27 who left the company: 20


## 6. Calculate the average age of employees who have more than 2 years of experience


In [ ]:
q6 = df.filter(col("ExperienceInCurrentDomain") > 2) \
       .agg(round(avg("Age"), 2).alias("AverageAge"))

q6.show()


+----------+
|AverageAge|
+----------+
|     28.25|
+----------+



## 7. How many male employees who have more than 2 years of experience stayed in the company?


In [ ]:
# LeaveOrNot = 0 means stayed in the company
q7 = df.filter((col("Gender") == "Male") & \
               (col("ExperienceInCurrentDomain") > 2) & \
               (col("LeaveOrNot") == 0)).count()

print("Number of male employees with more than 2 years experience who stayed:", q7)


Number of male employees with more than 2 years experience who stayed: 37


## Optional: SQL Version


In [ ]:
df.createOrReplaceTempView("employees")

spark.sql("""
SELECT City, COUNT(*) AS NumberOfEmployees
FROM employees
GROUP BY City
ORDER BY City
""").show()

spark.sql("""
SELECT Education, ROUND(AVG(ExperienceInCurrentDomain), 2) AS AvgExperienceInCurrentDomain
FROM employees
GROUP BY Education
ORDER BY Education
""").show()

spark.sql("""
SELECT City, ROUND(AVG(Age), 2) AS AverageAge
FROM employees
GROUP BY City
ORDER BY City
""").show()

spark.sql("""
SELECT COUNT(*) AS FemaleEmployees_2YearsOrMore
FROM employees
WHERE Gender = 'Female' AND ExperienceInCurrentDomain >= 2
""").show()

spark.sql("""
SELECT COUNT(*) AS EmployeesOlderThan27WhoLeft
FROM employees
WHERE Age > 27 AND LeaveOrNot = 1
""").show()

spark.sql("""
SELECT ROUND(AVG(Age), 2) AS AverageAge
FROM employees
WHERE ExperienceInCurrentDomain > 2
""").show()

spark.sql("""
SELECT COUNT(*) AS MaleEmployeesMoreThan2YearsStayed
FROM employees
WHERE Gender = 'Male' AND ExperienceInCurrentDomain > 2 AND LeaveOrNot = 0
""").show()


+---------+-----------------+
|     City|NumberOfEmployees|
+---------+-----------------+
|Bangalore|               73|
|New Delhi|               40|
|     Pune|               37|
+---------+-----------------+

+---------+----------------------------+
|Education|AvgExperienceInCurrentDomain|
+---------+----------------------------+
|Bachelors|                        2.45|
|  Masters|                        2.91|
|      PHD|                        3.11|
+---------+----------------------------+

+---------+----------+
|     City|AverageAge|
+---------+----------+
|Bangalore|     29.38|
|New Delhi|     29.15|
|     Pune|     29.11|
+---------+----------+

+----------------------------+
|FemaleEmployees_2YearsOrMore|
+----------------------------+
|                          38|
+----------------------------+

+---------------------------+
|EmployeesOlderThan27WhoLeft|
+---------------------------+
|                         20|
+---------------------------+

+----------+
|AverageAge|
+-----